# NGS Fundamentals: From Sequencing to Alignment

## Tier 3 - Applied Bioinformatics

This notebook covers the complete NGS analysis pipeline: sequencing technologies, raw data formats, quality control, read trimming, alignment, and working with SAM/BAM files.

### Learning Objectives
- Understand how Illumina, PacBio, and Oxford Nanopore sequencing work
- Parse and analyze FASTQ files
- Interpret quality scores and FastQC reports
- Understand read trimming concepts (adapters, quality trimming)
- Work with SAM/BAM format: fields, flags, CIGAR strings
- Use samtools for basic BAM operations
- Calculate genome coverage and depth

---

---[< Previous: Ready for Applied Bioinformatics?](../00_Skills_Check/00_skills_check.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Bioinformatics Data Formats: A Comprehensive Guide >](02_bio_data_formats.ipynb)---

## 1. Sequencing Technologies

### 1.1 Illumina (Sequencing by Synthesis)

Illumina is the dominant short-read sequencing platform, used in the vast majority of genomics studies.

**How it works:**
1. **Library preparation**: DNA is fragmented, adapters are ligated to both ends
2. **Cluster generation**: Fragments bind to a flow cell, are amplified by bridge PCR into clonal clusters (~1000 copies each)
3. **Sequencing by synthesis**: Fluorescently labeled, reversibly terminated nucleotides are added one at a time. After each incorporation, the flow cell is imaged to identify the base. The terminator is then cleaved, allowing the next cycle.
4. **Base calling**: Images are processed to call bases and assign quality scores

**Key characteristics:**
- Read length: 50-300 bp (paired-end: 2 x 150 bp most common)
- Error rate: ~0.1% (substitution errors dominate)
- Throughput: Up to 6 Tb per run (NovaSeq 6000)
- Cost: Lowest per-base cost

```
Illumina Sequencing by Synthesis:

Flow cell surface
    |  Bridge PCR         Clonal cluster
    |  ~~~>~~~>~~~   ->   |||||||||||||
    |                     |||||||||||||
    
Cycle 1:   A-fluor  ->  Image  ->  Cleave terminator
Cycle 2:   T-fluor  ->  Image  ->  Cleave terminator
Cycle 3:   G-fluor  ->  Image  ->  Cleave terminator
  ...       ...          ...          ...
```

### 1.2 PacBio (Single Molecule Real Time - SMRT)

PacBio produces long reads by observing a single polymerase incorporating nucleotides in real time.

**How it works:**
1. **SMRTbell library**: Circular DNA template with hairpin adapters on both ends
2. **Zero-mode waveguide (ZMW)**: A single polymerase sits at the bottom of a tiny well. As it incorporates fluorescently labeled nucleotides, the fluorescence is detected in real time.
3. **Circular consensus**: The polymerase reads around the circular template multiple times, generating subreads that are combined into a high-accuracy HiFi read.

**Key characteristics:**
- CLR (Continuous Long Read): 10-25 kb, ~10-15% error
- HiFi (Circular Consensus): 10-25 kb, >99.9% accuracy (Q30+)
- Throughput: ~30 Gb per SMRT cell (Revio)
- Errors: Insertions dominate (random, not systematic)

```
PacBio SMRT Sequencing:

  SMRTbell template (circular):
  
  5'---ACGTACGT---3'
  3'---TGCATGCA---5'
      ^         ^
   hairpin   hairpin
   adapter   adapter

  Polymerase reads around and around:
  Pass 1: ------>  (subread 1)
  Pass 2: <------  (subread 2)
  Pass 3: ------>  (subread 3)
  
  Consensus of passes = HiFi read (Q30+)
```

### 1.3 Oxford Nanopore (Nanopore Sequencing)

Oxford Nanopore passes DNA through a protein nanopore and measures changes in ionic current to identify bases.

**How it works:**
1. **Library preparation**: Adapters with motor proteins are ligated to DNA
2. **Nanopore translocation**: A motor protein feeds the DNA strand through a protein nanopore embedded in a membrane. As each base (or k-mer) passes through the pore, it partially blocks the ionic current in a characteristic way.
3. **Base calling**: The current signal is decoded by neural networks (e.g., Guppy, Dorado) into a DNA sequence.

**Key characteristics:**
- Read length: Theoretically unlimited (>1 Mb demonstrated)
- Typical: 10-100 kb, depending on DNA integrity
- Error rate: ~5% raw (R9.4.1), ~1% with R10.4.1 + latest basecallers
- Real-time: Data available as sequencing runs
- Portable: MinION is a USB-powered device

```
Oxford Nanopore Sequencing:

  Membrane
  ==========[PORE]==========
             | |
    Motor -> A T G C A T ...
    protein  | |
             v v
  
  Current signal:
  ___/\___/\_____/\__/\___
    A   T   G  C  A  T
  
  Neural network -> base calls
```

### 1.4 Platform Comparison

| Feature | Illumina | PacBio HiFi | Oxford Nanopore |
|---------|----------|-------------|------------------|
| Read length | 50-300 bp | 10-25 kb | 10 kb - 1 Mb+ |
| Accuracy | ~99.9% | ~99.9% | ~99% (R10.4.1 simplex), ~99.9% (duplex) |
| Error type | Substitutions | Random substitutions (CCS averages out CLR indels) | Homopolymer indels (R9), substitutions (R10.4.1) |
| Throughput | Up to 6 Tb/run | ~30 Gb/cell | 50-200 Gb/cell |
| Time to data | 1-3 days | Hours | Real-time |
| Cost/Gb | Lowest | Medium | Medium |
| Best for | WGS, RNA-seq, ChIP-seq | Genome assembly, SVs | Structural variants, field work |

**Choosing a platform:**
- **High coverage WGS, RNA-seq**: Illumina (cost-effective, high accuracy)
- **De novo assembly, structural variants**: PacBio HiFi (long + accurate)
- **Rapid diagnostics, ultra-long reads**: Oxford Nanopore (portable, real-time)
- **Hybrid approaches**: Combine short and long reads for best results

---

## 2. FASTQ Format

FASTQ is the standard format for storing sequencing reads with per-base quality scores.

### 2.1 Structure

Each read occupies exactly 4 lines:

```
@SEQ_ID                           <- Line 1: Header (starts with @), read identifier
GATCGATCGATCGATCGATC              <- Line 2: DNA sequence
+                                 <- Line 3: Separator (+ optionally followed by ID)
IIIIIIIIIHHHHHGGGFF               <- Line 4: Quality scores (ASCII encoded, same length as seq)
```

### 2.2 Quality Scores (Phred+33)

Quality scores encode the probability that a base call is wrong:

**Phred score** = -10 * log10(P_error)

| Phred Score | Error Probability | Accuracy | ASCII Character |
|-------------|-------------------|----------|------------------|
| 0 | 1.0 (100%) | 0% | ! (33) |
| 10 | 0.1 (10%) | 90% | + (43) |
| 20 | 0.01 (1%) | 99% | 5 (53) |
| 30 | 0.001 (0.1%) | 99.9% | ? (63) |
| 40 | 0.0001 (0.01%) | 99.99% | I (73) |

**Encoding**: ASCII character value - 33 = Phred score (Phred+33 / Sanger encoding, used by all modern platforms)

In [ ]:
import math
import numpy as np
from collections import Counter, defaultdict

# Phred score utilities

def phred_to_error_prob(phred):
    """Convert Phred quality score to error probability."""
    return 10 ** (-phred / 10)

def error_prob_to_phred(prob):
    """Convert error probability to Phred quality score."""
    if prob <= 0:
        return 40  # cap at Q40
    return -10 * math.log10(prob)

def ascii_to_phred(char, offset=33):
    """Convert ASCII quality character to Phred score."""
    return ord(char) - offset

def phred_to_ascii(phred, offset=33):
    """Convert Phred score to ASCII quality character."""
    return chr(phred + offset)

# Demonstrate the encoding
print("Phred Score Encoding (Phred+33):")
print(f"{'Phred':>6} {'ASCII':>6} {'Error Prob':>12} {'Accuracy':>10}")
print("-" * 40)
for q in [0, 5, 10, 15, 20, 25, 30, 35, 40]:
    p_err = phred_to_error_prob(q)
    print(f"{q:>6} {phred_to_ascii(q):>6} {p_err:>12.6f} {(1 - p_err)*100:>9.4f}%")

# Decode an example quality string
qual_string = "IIIII?55!!##FFFF"
print(f"\nDecoding quality string: {qual_string}")
scores = [ascii_to_phred(c) for c in qual_string]
print(f"Phred scores: {scores}")
print(f"Mean quality: Q{np.mean(scores):.1f}")

### 2.3 Paired-End Reads

Most Illumina experiments generate **paired-end** reads: two reads from opposite ends of the same DNA fragment.

```
DNA fragment:
5'----[====Read 1====>........<====Read 2====]----3'
      |<------------- insert size ------------->|

File naming convention:
  sample_R1.fastq.gz  (forward reads)
  sample_R2.fastq.gz  (reverse reads)
  
Reads are in the same order in both files:
  R1 read 1  <-->  R2 read 1  (same fragment)
  R1 read 2  <-->  R2 read 2  (same fragment)
  ...
```

**Insert size**: Distance between the start of Read 1 and the end of Read 2. Typical: 300-500 bp for standard libraries.

In [ ]:
def parse_fastq(filepath, max_reads=None):
    """
    Parse a FASTQ file and yield (header, sequence, quality_string) tuples.
    Handles both plain text and conceptually gzipped files.
    """
    count = 0
    with open(filepath, 'r') as f:
        while True:
            header = f.readline().strip()
            if not header:
                break
            sequence = f.readline().strip()
            f.readline()  # + line
            quality = f.readline().strip()
            yield (header[1:], sequence, quality)  # strip leading @
            count += 1
            if max_reads and count >= max_reads:
                break

# We'll work with simulated data for the examples
# In practice, you would use real FASTQ files

import random
random.seed(42)

def simulate_fastq_reads(n_reads=1000, read_length=150, mean_quality=30):
    """Generate simulated FASTQ reads for demonstration."""
    bases = 'ACGT'
    reads = []
    for i in range(n_reads):
        # Random sequence
        seq = ''.join(random.choice(bases) for _ in range(read_length))
        
        # Quality degrades toward end of read (realistic behavior)
        quals = []
        for pos in range(read_length):
            # Quality decreases with position
            base_q = mean_quality - (pos / read_length) * 10
            q = max(2, min(40, int(random.gauss(base_q, 4))))
            quals.append(q)
        
        qual_str = ''.join(phred_to_ascii(q) for q in quals)
        header = f"SIM:{i+1:06d} length={read_length}"
        reads.append((header, seq, qual_str))
    return reads

sim_reads = simulate_fastq_reads(n_reads=5000)

# Show first 3 reads
print("Simulated FASTQ reads (first 3):\n")
for header, seq, qual in sim_reads[:3]:
    scores = [ascii_to_phred(c) for c in qual]
    print(f"@{header}")
    print(f"{seq[:60]}...")
    print("+")
    print(f"{qual[:60]}...")
    print(f"  Mean Q: {np.mean(scores):.1f}\n")

---

## 3. Quality Control

### 3.1 FastQC: The Standard QC Tool

FastQC provides a comprehensive quality assessment of sequencing data. It produces a report with multiple modules:

```bash
# Install
# macOS: brew install fastqc
# Ubuntu: sudo apt install fastqc
# conda: conda install -c bioconda fastqc

# Run FastQC
fastqc sample_R1.fastq.gz sample_R2.fastq.gz -o qc_output/ -t 4

# Output: sample_R1_fastqc.html (visual report)
#         sample_R1_fastqc.zip  (raw data)
```

### 3.2 Key FastQC Metrics

| Module | What to Look For | Common Issues |
|--------|-----------------|---------------|
| Per-base quality | Q28+ across all positions | Quality drops at 3' end |
| Per-sequence quality | Peak at Q30+ | Bimodal = some reads failed |
| Per-base sequence content | Uniform A/T/G/C | Bias in first 10-15 bp (normal for RNA-seq) |
| GC content | Normal distribution matching genome | Shifted = contamination |
| Sequence length | Uniform (before trimming) | Variable after trimming |
| Duplication levels | Low (<20%) | High in PCR-heavy or targeted experiments |
| Overrepresented sequences | None or few | Adapter dimers, rRNA |
| Adapter content | <5% at read ends | >10% needs trimming |

In [ ]:
import matplotlib.pyplot as plt

def compute_per_position_quality(reads):
    """Compute per-position quality statistics (like FastQC)."""
    position_quals = defaultdict(list)
    
    for header, seq, qual in reads:
        for pos, qchar in enumerate(qual):
            position_quals[pos].append(ascii_to_phred(qchar))
    
    positions = sorted(position_quals.keys())
    stats = {
        'positions': positions,
        'median': [np.median(position_quals[p]) for p in positions],
        'q25': [np.percentile(position_quals[p], 25) for p in positions],
        'q75': [np.percentile(position_quals[p], 75) for p in positions],
        'p10': [np.percentile(position_quals[p], 10) for p in positions],
        'p90': [np.percentile(position_quals[p], 90) for p in positions],
    }
    return stats

# Compute and plot per-position quality
qual_stats = compute_per_position_quality(sim_reads)

fig, ax = plt.subplots(figsize=(14, 5))

pos = qual_stats['positions']
ax.fill_between(pos, qual_stats['p10'], qual_stats['p90'],
                alpha=0.15, color='blue', label='10th-90th percentile')
ax.fill_between(pos, qual_stats['q25'], qual_stats['q75'],
                alpha=0.3, color='blue', label='IQR (25th-75th)')
ax.plot(pos, qual_stats['median'], 'b-', linewidth=2, label='Median')

# Quality zones
ax.axhspan(30, 42, alpha=0.08, color='green')
ax.axhspan(20, 30, alpha=0.08, color='yellow')
ax.axhspan(0, 20, alpha=0.08, color='red')
ax.axhline(y=30, color='green', linestyle='--', alpha=0.5, label='Q30')
ax.axhline(y=20, color='orange', linestyle='--', alpha=0.5, label='Q20')

ax.set_xlabel('Position in Read (bp)')
ax.set_ylabel('Phred Quality Score')
ax.set_title('Per-Position Quality Scores (FastQC-style)')
ax.set_ylim(0, 42)
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
def compute_qc_summary(reads):
    """Generate a basic QC summary report."""
    total_reads = 0
    total_bases = 0
    gc_count = 0
    lengths = []
    mean_quals = []
    base_counts = Counter()
    
    for header, seq, qual in reads:
        total_reads += 1
        total_bases += len(seq)
        lengths.append(len(seq))
        gc_count += seq.count('G') + seq.count('C')
        base_counts.update(seq)
        
        scores = [ascii_to_phred(c) for c in qual]
        mean_quals.append(np.mean(scores))
    
    q20_reads = sum(1 for q in mean_quals if q >= 20)
    q30_reads = sum(1 for q in mean_quals if q >= 30)
    
    print("=" * 50)
    print("          FASTQ Quality Report")
    print("=" * 50)
    print(f"Total reads:       {total_reads:>12,}")
    print(f"Total bases:       {total_bases:>12,}")
    print(f"Read length:       {min(lengths)}-{max(lengths)} bp")
    print(f"Mean read length:  {np.mean(lengths):>12.1f} bp")
    print(f"GC content:        {gc_count/total_bases*100:>11.1f}%")
    print(f"Mean quality:      Q{np.mean(mean_quals):>10.1f}")
    print(f"Reads >= Q20:      {q20_reads:>8,} ({100*q20_reads/total_reads:.1f}%)")
    print(f"Reads >= Q30:      {q30_reads:>8,} ({100*q30_reads/total_reads:.1f}%)")
    print(f"\nBase composition:")
    for base in 'ACGTN':
        count = base_counts.get(base, 0)
        print(f"  {base}: {count:>10,} ({100*count/total_bases:.1f}%)")
    print("=" * 50)
    
    return mean_quals

mean_quals = compute_qc_summary(sim_reads)

In [ ]:
# Per-sequence quality distribution and GC content
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Per-sequence quality distribution
axes[0].hist(mean_quals, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=20, color='orange', linestyle='--', label='Q20')
axes[0].axvline(x=30, color='red', linestyle='--', label='Q30')
axes[0].set_xlabel('Mean Read Quality (Phred)')
axes[0].set_ylabel('Number of Reads')
axes[0].set_title('Per-Sequence Quality Distribution')
axes[0].legend()

# GC content per read
gc_per_read = []
for header, seq, qual in sim_reads:
    gc_frac = (seq.count('G') + seq.count('C')) / len(seq) * 100
    gc_per_read.append(gc_frac)

axes[1].hist(gc_per_read, bins=40, edgecolor='black', alpha=0.7, color='seagreen')
axes[1].set_xlabel('GC Content (%)')
axes[1].set_ylabel('Number of Reads')
axes[1].set_title('Per-Read GC Content Distribution')
axes[1].axvline(x=np.mean(gc_per_read), color='red', linestyle='--',
                label=f'Mean: {np.mean(gc_per_read):.1f}%')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3 Interpreting FastQC Reports

**Common patterns and their meaning:**

| Observation | Likely Cause | Action |
|-------------|-------------|--------|
| Quality drops at 3' end | Normal Illumina behavior | Trim low-quality tails |
| Adapter content >5% | Short inserts, read-through | Trim adapters |
| Bimodal GC distribution | Contamination with another organism | Investigate, filter reads |
| High duplication (>50%) | Low library complexity or PCR over-amplification | Mark/remove duplicates |
| Uneven base content at start | Random hexamer priming bias (RNA-seq) | Usually acceptable |
| Per-tile quality failures | Flowcell bubble or debris | Resequence if severe |
| Overrepresented sequences | Adapter dimers, rRNA, primer | Trim/filter |

**Not all failures are problems:** FastQC uses generic thresholds. For example:
- RNA-seq data often fails the "Per-base sequence content" module (hexamer bias is expected)
- Amplicon data will always fail the "Sequence duplication" module
- Bisulfite-seq will fail the "Per-base sequence content" module (C->T conversion)

---

## 4. Read Trimming

### 4.1 Why Trim?

Raw reads may contain:
- **Adapter sequences**: Technical sequences ligated during library prep that are not part of the original DNA
- **Low-quality bases**: Especially at the 3' end of reads
- **Poly-G tails**: Artifact from NextSeq/NovaSeq two-color chemistry

These need to be removed before alignment to avoid false mappings.

### 4.2 Trimming Tools

**Trimmomatic** (widely used, Java-based):
```bash
trimmomatic PE -phred33 \
  input_R1.fastq.gz input_R2.fastq.gz \
  output_R1_paired.fq.gz output_R1_unpaired.fq.gz \
  output_R2_paired.fq.gz output_R2_unpaired.fq.gz \
  ILLUMINACLIP:adapters.fa:2:30:10 \
  LEADING:3 \
  TRAILING:3 \
  SLIDINGWINDOW:4:15 \
  MINLEN:36
```

| Parameter | Meaning |
|-----------|--------|
| ILLUMINACLIP | Remove adapter sequences |
| LEADING:3 | Remove leading bases below quality 3 |
| TRAILING:3 | Remove trailing bases below quality 3 |
| SLIDINGWINDOW:4:15 | Scan 4-base window, cut when avg quality < 15 |
| MINLEN:36 | Drop reads shorter than 36 bp after trimming |

**fastp** (faster, more modern, auto-detects adapters):
```bash
fastp -i input_R1.fastq.gz -I input_R2.fastq.gz \
      -o output_R1.fastq.gz -O output_R2.fastq.gz \
      --detect_adapter_for_pe \
      --cut_tail --cut_tail_mean_quality 20 \
      --length_required 36 \
      --html report.html --json report.json
```

In [ ]:
def sliding_window_trim(sequence, quality_str, window_size=4, min_quality=20, min_length=36):
    """
    Implement sliding window quality trimming (like Trimmomatic).
    
    Scans from 5' to 3' with a window. When average quality in the
    window drops below min_quality, trim from that position onward.
    """
    quals = [ascii_to_phred(c) for c in quality_str]
    trim_pos = len(quals)
    
    for i in range(len(quals) - window_size + 1):
        window_avg = np.mean(quals[i:i + window_size])
        if window_avg < min_quality:
            trim_pos = i
            break
    
    trimmed_seq = sequence[:trim_pos]
    trimmed_qual = quality_str[:trim_pos]
    
    if len(trimmed_seq) < min_length:
        return None, None  # Read too short after trimming
    
    return trimmed_seq, trimmed_qual

def adapter_trim(sequence, quality_str, adapter='AGATCGGAAGAG', min_overlap=6):
    """
    Simple adapter trimming: find adapter at 3' end and remove it.
    Checks for partial matches (read-through into adapter).
    """
    # Check for full or partial adapter match at the end
    for start in range(len(sequence) - min_overlap + 1):
        remaining = sequence[start:]
        adapter_prefix = adapter[:len(remaining)]
        mismatches = sum(1 for a, b in zip(remaining, adapter_prefix) if a != b)
        if mismatches <= 1:  # allow 1 mismatch
            return sequence[:start], quality_str[:start]
    
    return sequence, quality_str

# Demonstrate trimming on simulated reads
print("Sliding Window Trimming Demo (window=4, min_quality=20):\n")

trimmed_count = 0
dropped_count = 0
original_lengths = []
trimmed_lengths = []

for header, seq, qual in sim_reads:
    original_lengths.append(len(seq))
    t_seq, t_qual = sliding_window_trim(seq, qual, window_size=4, min_quality=20)
    
    if t_seq is None:
        dropped_count += 1
    else:
        trimmed_lengths.append(len(t_seq))
        if len(t_seq) < len(seq):
            trimmed_count += 1

print(f"Total reads: {len(sim_reads)}")
print(f"Trimmed:     {trimmed_count} ({100*trimmed_count/len(sim_reads):.1f}%)")
print(f"Dropped:     {dropped_count} ({100*dropped_count/len(sim_reads):.1f}%)")
print(f"Surviving:   {len(trimmed_lengths)} ({100*len(trimmed_lengths)/len(sim_reads):.1f}%)")
if trimmed_lengths:
    print(f"\nLength before trimming: {np.mean(original_lengths):.0f} bp (all reads)")
    print(f"Length after trimming:  {np.mean(trimmed_lengths):.0f} bp (surviving reads)")

In [ ]:
# Visualize trimming effect
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Read length distribution after trimming
axes[0].hist(original_lengths, bins=30, alpha=0.5, label='Before', edgecolor='black')
axes[0].hist(trimmed_lengths, bins=30, alpha=0.5, label='After', edgecolor='black')
axes[0].set_xlabel('Read Length (bp)')
axes[0].set_ylabel('Number of Reads')
axes[0].set_title('Read Length: Before vs After Trimming')
axes[0].legend()

# Quality before and after
before_quals = [np.mean([ascii_to_phred(c) for c in qual]) for _, _, qual in sim_reads]
after_quals = []
for _, seq, qual in sim_reads:
    t_seq, t_qual = sliding_window_trim(seq, qual)
    if t_qual:
        after_quals.append(np.mean([ascii_to_phred(c) for c in t_qual]))

axes[1].hist(before_quals, bins=30, alpha=0.5, label='Before', edgecolor='black')
axes[1].hist(after_quals, bins=30, alpha=0.5, label='After', edgecolor='black')
axes[1].set_xlabel('Mean Read Quality (Phred)')
axes[1].set_ylabel('Number of Reads')
axes[1].set_title('Mean Quality: Before vs After Trimming')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 5. Read Alignment

### 5.1 Alignment Concepts

After quality control, reads are aligned (mapped) to a reference genome to determine where each read originated.

```
Reference genome:
  ...ACGTACGTACGTACGTACGTACGTACGT...
      ||||||||||||
Read: ACGTACGTACGT  -> mapped at position X
```

**Challenges:**
- Genome is huge (3.2 billion bp for human)
- Millions to billions of reads
- Must handle mismatches, insertions, deletions
- Repetitive regions: some reads map to multiple locations

### 5.2 Popular Aligners

| Aligner | Type | Best For | Algorithm |
|---------|------|----------|----------|
| **BWA-MEM** | DNA, short reads | WGS, WES, ChIP-seq | BWT + seed-and-extend |
| **Bowtie2** | DNA, short reads | WGS, ChIP-seq | BWT + FM-index |
| **HISAT2** | RNA, splice-aware | RNA-seq | Graph FM-index |
| **STAR** | RNA, splice-aware | RNA-seq | Suffix array |
| **minimap2** | Long reads | PacBio, Nanopore | Minimizer-based |

### 5.3 Alignment Workflow

```bash
# Step 1: Index the reference genome (one-time step)
bwa index reference.fasta

# Step 2: Align reads to reference
bwa mem -t 8 reference.fasta reads_R1.fq.gz reads_R2.fq.gz > aligned.sam

# Step 3: Convert SAM to BAM, sort, and index
samtools view -bS aligned.sam | samtools sort -o aligned.sorted.bam
samtools index aligned.sorted.bam

# Or as a pipe (more efficient):
bwa mem -t 8 ref.fa R1.fq.gz R2.fq.gz \
  | samtools sort -o aligned.sorted.bam
samtools index aligned.sorted.bam
```

**Why splice-aware aligners for RNA-seq?**

```
Genomic DNA:   ---[Exon1]----intron (10 kb)----[Exon2]---
mRNA:          ---[Exon1][Exon2]---
RNA-seq read:  ---[ExonEEEEEEEExon]---  (spans junction)

BWA: Cannot handle the split -> unmapped or wrong
HISAT2/STAR: Recognizes exon-exon junctions -> correct alignment
```

---

## 6. SAM/BAM Format

### 6.1 Overview

- **SAM** (Sequence Alignment Map): Human-readable text format
- **BAM**: Binary compressed version (much smaller, used in practice)
- **CRAM**: Even more compressed (reference-based compression)

### 6.2 SAM Header

Lines starting with `@`:
```
@HD  VN:1.6  SO:coordinate           <- Header: version, sort order
@SQ  SN:chr1  LN:248956422           <- Sequence: name, length
@SQ  SN:chr2  LN:242193529
@RG  ID:sample1  SM:sample1  PL:ILLUMINA  <- Read group
@PG  ID:bwa  PN:bwa  VN:0.7.17      <- Program used
```

### 6.3 Alignment Records (11 Mandatory Fields)

| Col | Field | Type | Description |
|-----|-------|------|-------------|
| 1 | QNAME | String | Read/query name |
| 2 | FLAG | Int | Bitwise flags (see below) |
| 3 | RNAME | String | Reference sequence name (e.g., chr1) |
| 4 | POS | Int | 1-based leftmost mapping position |
| 5 | MAPQ | Int | Mapping quality (0-60) |
| 6 | CIGAR | String | Alignment description |
| 7 | RNEXT | String | Reference name of mate (= if same chr) |
| 8 | PNEXT | Int | Position of mate |
| 9 | TLEN | Int | Template/insert length |
| 10 | SEQ | String | Read sequence |
| 11 | QUAL | String | Base quality (Phred+33, same as FASTQ) |

### 6.4 SAM Flags

The FLAG field is a bitwise combination of properties:

| Bit | Decimal | Meaning |
|-----|---------|--------|
| 0x1 | 1 | Read is paired |
| 0x2 | 2 | Read is mapped in proper pair |
| 0x4 | 4 | Read is unmapped |
| 0x8 | 8 | Mate is unmapped |
| 0x10 | 16 | Read is on reverse strand |
| 0x20 | 32 | Mate is on reverse strand |
| 0x40 | 64 | First in pair (R1) |
| 0x80 | 128 | Second in pair (R2) |
| 0x100 | 256 | Secondary alignment |
| 0x200 | 512 | Fails quality checks |
| 0x400 | 1024 | PCR or optical duplicate |
| 0x800 | 2048 | Supplementary alignment |

**Common FLAG values:**
- **99** = 1+2+32+64 = paired, proper pair, mate reverse, first in pair (typical R1 forward)
- **147** = 1+2+16+128 = paired, proper pair, reverse strand, second in pair (typical R2 reverse)
- **4** = unmapped read
- **1024** = PCR duplicate

In [ ]:
FLAG_BITS = {
    0x1:   'paired',
    0x2:   'proper_pair',
    0x4:   'unmapped',
    0x8:   'mate_unmapped',
    0x10:  'reverse_strand',
    0x20:  'mate_reverse',
    0x40:  'first_in_pair',
    0x80:  'second_in_pair',
    0x100: 'secondary',
    0x200: 'failed_qc',
    0x400: 'duplicate',
    0x800: 'supplementary',
}

def interpret_flag(flag):
    """Decode SAM FLAG bitfield into human-readable components."""
    return [desc for bit, desc in sorted(FLAG_BITS.items()) if flag & bit]

def compose_flag(**kwargs):
    """Compose a SAM FLAG from named components."""
    name_to_bit = {v: k for k, v in FLAG_BITS.items()}
    flag = 0
    for name, active in kwargs.items():
        if active and name in name_to_bit:
            flag |= name_to_bit[name]
    return flag

# Decode common flags
print("Common SAM FLAG values:\n")
for flag in [99, 147, 83, 163, 4, 77, 141, 1024, 2048]:
    components = interpret_flag(flag)
    print(f"  FLAG {flag:>5} = {' + '.join(components)}")

# Compose a flag
print(f"\nCompose: paired + proper_pair + mate_reverse + first_in_pair")
print(f"  = FLAG {compose_flag(paired=True, proper_pair=True, mate_reverse=True, first_in_pair=True)}")

### 6.5 CIGAR Strings

CIGAR (Compact Idiosyncratic Gapped Alignment Report) describes how a read aligns to the reference:

| Op | Description | Consumes Query | Consumes Reference |
|----|------------|----------------|--------------------|
| M | Match/mismatch | Yes | Yes |
| I | Insertion to reference | Yes | No |
| D | Deletion from reference | No | Yes |
| N | Skipped region (intron in RNA-seq) | No | Yes |
| S | Soft clip (present in SEQ but not aligned) | Yes | No |
| H | Hard clip (not in SEQ) | No | No |
| = | Sequence match | Yes | Yes |
| X | Sequence mismatch | Yes | Yes |

**Examples:**
```
100M           -> 100 bases aligned (matches + mismatches)
50M2I48M       -> 50 match, 2-base insertion, 48 match
75M2D25M       -> 75 match, 2-base deletion, 25 match  
50M10000N50M   -> RNA-seq: 50 match, 10kb intron, 50 match
5S95M          -> 5 bases soft-clipped at start, 95 aligned
```

In [ ]:
import re

def parse_cigar(cigar_string):
    """Parse CIGAR string into list of (length, operation) tuples."""
    return [(int(length), op) for length, op in re.findall(r'(\d+)([MIDNSHP=X])', cigar_string)]

def cigar_stats(cigar_string):
    """Calculate alignment statistics from CIGAR string."""
    ops = parse_cigar(cigar_string)
    
    query_consuming = {'M', 'I', 'S', '=', 'X'}  # consume read bases
    ref_consuming = {'M', 'D', 'N', '=', 'X'}     # consume reference bases
    
    read_length = sum(n for n, op in ops if op in query_consuming)
    ref_span = sum(n for n, op in ops if op in ref_consuming)
    aligned_bases = sum(n for n, op in ops if op in {'M', '=', 'X'})
    insertions = sum(n for n, op in ops if op == 'I')
    deletions = sum(n for n, op in ops if op == 'D')
    soft_clipped = sum(n for n, op in ops if op == 'S')
    introns = sum(n for n, op in ops if op == 'N')
    
    return {
        'read_length': read_length,
        'ref_span': ref_span,
        'aligned_bases': aligned_bases,
        'insertions': insertions,
        'deletions': deletions,
        'soft_clipped': soft_clipped,
        'introns': introns,
    }

def visualize_cigar(cigar_string, max_width=80):
    """Create a text visualization of a CIGAR alignment."""
    ops = parse_cigar(cigar_string)
    ref_line = []
    query_line = []
    match_line = []
    
    for n, op in ops:
        display_n = min(n, 20)  # limit display width for large operations
        suffix = f'..({n})' if n > 20 else ''
        
        if op == 'M' or op == '=' or op == 'X':
            ref_line.append('R' * display_n + suffix)
            query_line.append('Q' * display_n + suffix)
            match_line.append('|' * display_n + suffix)
        elif op == 'I':
            ref_line.append('-' * display_n)
            query_line.append('I' * display_n)
            match_line.append(' ' * display_n)
        elif op == 'D':
            ref_line.append('R' * display_n + suffix)
            query_line.append('-' * display_n + suffix)
            match_line.append(' ' * display_n + suffix)
        elif op == 'N':
            ref_line.append(f'~~~intron({n})~~~')
            query_line.append(f'~~~  skip  ~~~')
            match_line.append(f'              ')
        elif op == 'S':
            ref_line.append(' ' * display_n)
            query_line.append('s' * display_n)
            match_line.append(' ' * display_n)
    
    print(f"CIGAR: {cigar_string}")
    print(f"Ref:   {''.join(ref_line)[:max_width]}")
    print(f"       {''.join(match_line)[:max_width]}")
    print(f"Read:  {''.join(query_line)[:max_width]}")

# Examples
examples = [
    ('100M', 'Perfect alignment (100 bp)'),
    ('50M2I48M', '50 match, 2-bp insertion, 48 match'),
    ('75M2D25M', '75 match, 2-bp deletion, 25 match'),
    ('50M10000N50M', 'RNA-seq spliced read (10 kb intron)'),
    ('5S95M', '5 bp soft-clipped at start'),
]

for cigar, description in examples:
    print(f"\n{description}")
    print("-" * 60)
    visualize_cigar(cigar)
    stats = cigar_stats(cigar)
    print(f"  Read length: {stats['read_length']} bp, Ref span: {stats['ref_span']} bp, "
          f"Aligned: {stats['aligned_bases']} bp")

In [ ]:
def parse_sam_line(line):
    """Parse a single SAM alignment record."""
    fields = line.strip().split('\t')
    if len(fields) < 11:
        return None
    return {
        'qname': fields[0],
        'flag': int(fields[1]),
        'rname': fields[2],
        'pos': int(fields[3]) if fields[3] != '*' else 0,
        'mapq': int(fields[4]),
        'cigar': fields[5],
        'rnext': fields[6],
        'pnext': int(fields[7]) if fields[7] != '*' else 0,
        'tlen': int(fields[8]),
        'seq': fields[9],
        'qual': fields[10],
        'tags': fields[11:] if len(fields) > 11 else []
    }

# Create example SAM records for demonstration
example_sam = """@HD\tVN:1.6\tSO:coordinate
@SQ\tSN:chr1\tLN:248956422
@SQ\tSN:chr2\tLN:242193529
@PG\tID:bwa\tPN:bwa\tVN:0.7.17
read001\t99\tchr1\t10000\t60\t100M\t=\t10200\t300\tACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGT\tIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII\tNM:i:0\tMD:Z:100
read001\t147\tchr1\t10200\t60\t100M\t=\t10000\t-300\tACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGT\tIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII\tNM:i:0\tMD:Z:100
read002\t99\tchr1\t15000\t45\t50M2I48M\t=\t15250\t350\tACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGT\tIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII\tNM:i:2\tMD:Z:98
read003\t4\t*\t0\t0\t*\t*\t0\t0\tACGTACGTACGT\tIIIIIIIIIIII\tNM:i:0
read004\t99\tchr1\t20000\t20\t75M3D25M\t=\t20200\t300\tACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGT\tIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII\tNM:i:3\tMD:Z:75^ACG25
read005\t1024\tchr1\t10000\t60\t100M\t=\t10200\t300\tACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGT\tIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII\tNM:i:0\tMD:Z:100"""

# Parse and analyze
headers = []
alignments = []
for line in example_sam.strip().split('\n'):
    if line.startswith('@'):
        headers.append(line)
    else:
        aln = parse_sam_line(line)
        if aln:
            alignments.append(aln)

print(f"Header lines: {len(headers)}")
print(f"Alignment records: {len(alignments)}\n")

for aln in alignments:
    flags = interpret_flag(aln['flag'])
    print(f"{aln['qname']}:")
    print(f"  Position: {aln['rname']}:{aln['pos']}")
    print(f"  FLAG {aln['flag']}: {', '.join(flags) if flags else 'mapped forward'}")
    print(f"  MAPQ: {aln['mapq']}")
    print(f"  CIGAR: {aln['cigar']}")
    if aln['cigar'] != '*':
        stats = cigar_stats(aln['cigar'])
        print(f"  Ref span: {stats['ref_span']} bp, Insertions: {stats['insertions']}, Deletions: {stats['deletions']}")
    print()

---

## 7. Samtools Basics

Samtools is the essential toolkit for working with SAM/BAM files.

### 7.1 Core Commands

```bash
# Convert SAM to BAM
samtools view -bS input.sam -o output.bam

# Sort BAM by coordinate (required for most downstream tools)
samtools sort input.bam -o sorted.bam

# Index BAM (creates .bai file, required for random access)
samtools index sorted.bam

# View alignment statistics
samtools flagstat sorted.bam

# View detailed statistics
samtools stats sorted.bam | grep ^SN

# Count reads in a region
samtools view -c sorted.bam chr1:1000000-2000000

# Extract reads from a region
samtools view sorted.bam chr1:1000000-2000000 | head

# Filter by mapping quality
samtools view -b -q 30 sorted.bam -o high_quality.bam

# Filter: only properly paired reads
samtools view -b -f 2 sorted.bam -o proper_pairs.bam

# Filter: exclude unmapped reads
samtools view -b -F 4 sorted.bam -o mapped.bam

# Calculate depth at each position
samtools depth sorted.bam > depth.txt

# Mark PCR duplicates (newer samtools)
samtools markdup sorted.bam dedup.bam
```

### 7.2 Understanding samtools flagstat Output

```
1000000 + 0 in total (QC-passed reads + QC-failed reads)
  50000 + 0 secondary
      0 + 0 supplementary
  10000 + 0 duplicates
 980000 + 0 mapped (98.00% : N/A)
 950000 + 0 paired in sequencing
 475000 + 0 read1
 475000 + 0 read2
 920000 + 0 properly paired (96.84% : N/A)
 940000 + 0 with itself and mate mapped
  10000 + 0 singletons (1.05% : N/A)
   5000 + 0 with mate mapped to a different chr
   2000 + 0 with mate mapped to a different chr (mapQ>=5)
```

In [ ]:
def flagstat_summary(alignments):
    """Generate flagstat-like summary from parsed alignments."""
    total = len(alignments)
    mapped = sum(1 for a in alignments if not (a['flag'] & 0x4))
    unmapped = sum(1 for a in alignments if a['flag'] & 0x4)
    paired = sum(1 for a in alignments if a['flag'] & 0x1)
    proper_pair = sum(1 for a in alignments if a['flag'] & 0x2)
    duplicates = sum(1 for a in alignments if a['flag'] & 0x400)
    secondary = sum(1 for a in alignments if a['flag'] & 0x100)
    supplementary = sum(1 for a in alignments if a['flag'] & 0x800)
    read1 = sum(1 for a in alignments if a['flag'] & 0x40)
    read2 = sum(1 for a in alignments if a['flag'] & 0x80)
    
    print("=" * 50)
    print("          Flagstat Summary")
    print("=" * 50)
    print(f"{total:>10} total reads")
    print(f"{secondary:>10} secondary")
    print(f"{supplementary:>10} supplementary")
    print(f"{duplicates:>10} duplicates")
    print(f"{mapped:>10} mapped ({100*mapped/total:.1f}%)" if total else "")
    print(f"{paired:>10} paired in sequencing")
    print(f"{read1:>10} read1")
    print(f"{read2:>10} read2")
    print(f"{proper_pair:>10} properly paired ({100*proper_pair/paired:.1f}%)" if paired else "")
    print(f"{unmapped:>10} unmapped")
    print("=" * 50)

flagstat_summary(alignments)

In [ ]:
# Simulate a larger set of alignments for statistics
random.seed(42)

def simulate_alignments(n=10000, genome_size=1_000_000, read_length=150):
    """Simulate alignment data for demonstration."""
    alns = []
    for i in range(n):
        pos = random.randint(1, genome_size - read_length)
        mapq = random.choices([0, 10, 20, 30, 40, 50, 60],
                              weights=[5, 3, 5, 10, 15, 20, 42])[0]
        # Most reads map well
        is_unmapped = random.random() < 0.02
        flag = 99  # default: paired, proper, mate_reverse, first_in_pair
        if is_unmapped:
            flag = 4
            mapq = 0
            pos = 0
        
        cigar_choices = ['150M', '100M1I49M', '75M2D75M', '5S145M', '145M5S']
        cigar_weights = [80, 5, 5, 5, 5]
        cigar = random.choices(cigar_choices, weights=cigar_weights)[0] if not is_unmapped else '*'
        
        alns.append({
            'qname': f'read{i:06d}',
            'flag': flag,
            'rname': 'chr1' if not is_unmapped else '*',
            'pos': pos,
            'mapq': mapq,
            'cigar': cigar,
        })
    return alns

sim_alns = simulate_alignments(10000)

# MAPQ distribution
mapqs = [a['mapq'] for a in sim_alns if not (a['flag'] & 0x4)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(mapqs, bins=range(0, 65, 2), edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=30, color='red', linestyle='--', label='MAPQ 30 threshold')
axes[0].set_xlabel('Mapping Quality (MAPQ)')
axes[0].set_ylabel('Number of Reads')
axes[0].set_title('Mapping Quality Distribution')
axes[0].legend()

# CIGAR operation distribution
cigar_types = Counter()
for a in sim_alns:
    if a['cigar'] != '*':
        ops = parse_cigar(a['cigar'])
        for n, op in ops:
            if op != 'M':
                cigar_types[op] += 1

if cigar_types:
    ops_sorted = sorted(cigar_types.items(), key=lambda x: x[1], reverse=True)
    axes[1].bar([o for o, c in ops_sorted], [c for o, c in ops_sorted],
               edgecolor='black', alpha=0.7, color='coral')
    axes[1].set_xlabel('CIGAR Operation')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Non-M CIGAR Operations')

plt.tight_layout()
plt.show()

high_mapq = sum(1 for q in mapqs if q >= 30)
print(f"Reads with MAPQ >= 30: {high_mapq} ({100*high_mapq/len(mapqs):.1f}%)")
print(f"Multi-mapped (MAPQ=0): {mapqs.count(0)} ({100*mapqs.count(0)/len(mapqs):.1f}%)")

---

## 8. Genome Coverage and Depth

### 8.1 Definitions

- **Depth (at a position)**: Number of reads covering that position
- **Coverage (breadth)**: Fraction of the genome covered by at least one read
- **Mean depth**: Average number of reads across all positions

### 8.2 Calculating Coverage

**Lander-Waterman formula** (theoretical expectation):

```
Mean depth = (N * L) / G

Where:
  N = number of reads
  L = read length
  G = genome size
  
Coverage (breadth) = 1 - e^(-depth)
```

**Example for 30x human WGS:**
- Genome = 3.2 Gb, Read length = 150 bp, Depth target = 30x
- Reads needed = 30 * 3.2e9 / 150 = 640 million reads
- Paired-end: 320 million read pairs

In [ ]:
def lander_waterman(n_reads, read_length, genome_size):
    """Calculate expected coverage using Lander-Waterman model."""
    mean_depth = (n_reads * read_length) / genome_size
    breadth = 1 - math.exp(-mean_depth)
    return mean_depth, breadth

def reads_for_depth(target_depth, read_length, genome_size):
    """Calculate number of reads needed for a target depth."""
    return int((target_depth * genome_size) / read_length)

# Human WGS examples
human_genome = 3.2e9  # 3.2 billion bp
read_len = 150

print("Human WGS Coverage Calculations (2x150 bp reads)")
print("=" * 60)
print(f"{'Target Depth':>14} {'Reads Needed':>14} {'Read Pairs':>14} {'Breadth':>10}")
print("-" * 60)

for target in [1, 5, 10, 30, 50, 100]:
    n_reads = reads_for_depth(target, read_len, human_genome)
    depth, breadth = lander_waterman(n_reads, read_len, human_genome)
    print(f"{target:>12}x {n_reads:>14,} {n_reads//2:>14,} {breadth*100:>9.2f}%")

# Data size estimates
print("\nApproximate data sizes (compressed FASTQ):")
for depth_target in [30, 50]:
    n = reads_for_depth(depth_target, read_len, human_genome)
    # ~0.5 bytes per base compressed
    size_gb = (n * read_len * 0.5) / 1e9
    print(f"  {depth_target}x WGS: ~{n/1e6:.0f}M reads, ~{size_gb:.0f} GB compressed")

In [ ]:
def simulate_coverage(n_reads, read_length, genome_size):
    """Simulate read placement and calculate per-position depth."""
    depth = np.zeros(genome_size, dtype=np.int32)
    
    for _ in range(n_reads):
        start = random.randint(0, genome_size - read_length)
        depth[start:start + read_length] += 1
    
    return depth

# Simulate coverage on a small "genome"
small_genome = 10000
n_reads_sim = 2000
rl = 150

depth_array = simulate_coverage(n_reads_sim, rl, small_genome)

expected_depth = (n_reads_sim * rl) / small_genome
actual_mean = np.mean(depth_array)
breadth = np.sum(depth_array > 0) / small_genome * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Depth across genome
axes[0].plot(depth_array, linewidth=0.5, color='steelblue')
axes[0].axhline(y=actual_mean, color='red', linestyle='--',
                label=f'Mean depth: {actual_mean:.1f}x')
axes[0].axhline(y=expected_depth, color='green', linestyle=':',
                label=f'Expected (Lander-Waterman): {expected_depth:.1f}x')
axes[0].set_xlabel('Genomic Position')
axes[0].set_ylabel('Read Depth')
axes[0].set_title(f'Coverage Profile ({n_reads_sim} reads x {rl} bp on {small_genome} bp genome)')
axes[0].legend()

# Depth distribution (histogram)
axes[1].hist(depth_array, bins=range(0, int(max(depth_array)) + 2),
             edgecolor='black', alpha=0.7, color='steelblue', density=True)

# Overlay Poisson distribution (theoretical)
from scipy.stats import poisson
x = np.arange(0, int(max(depth_array)) + 1)
axes[1].plot(x, poisson.pmf(x, expected_depth), 'r-o', markersize=3,
             label=f'Poisson(lambda={expected_depth:.1f})')
axes[1].set_xlabel('Depth')
axes[1].set_ylabel('Fraction of Positions')
axes[1].set_title('Depth Distribution vs Poisson')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Genome size: {small_genome:,} bp")
print(f"Reads: {n_reads_sim:,} x {rl} bp")
print(f"Expected mean depth: {expected_depth:.1f}x")
print(f"Actual mean depth: {actual_mean:.1f}x")
print(f"Genome breadth (>0 depth): {breadth:.1f}%")
print(f"Positions with >=10x: {np.sum(depth_array >= 10)/small_genome*100:.1f}%")
print(f"Positions with 0x: {np.sum(depth_array == 0)/small_genome*100:.2f}%")

---

## 9. Practical: Working with BioPython for FASTQ

BioPython's `SeqIO` module provides a convenient interface for parsing FASTQ files.

In [ ]:
# BioPython FASTQ parsing
# In practice, use: from Bio import SeqIO
# records = SeqIO.parse("reads.fastq", "fastq")

# Here we demonstrate the BioPython API with a simulated FASTQ file
import tempfile
import os

# Write simulated reads to a temporary FASTQ file
tmpfile = tempfile.NamedTemporaryFile(mode='w', suffix='.fastq', delete=False)
for header, seq, qual in sim_reads[:100]:
    tmpfile.write(f"@{header}\n{seq}\n+\n{qual}\n")
tmpfile.close()

try:
    from Bio import SeqIO
    
    # Parse with BioPython
    records = list(SeqIO.parse(tmpfile.name, 'fastq'))
    print(f"Parsed {len(records)} reads with BioPython\n")
    
    # Inspect first record
    rec = records[0]
    print(f"Record ID: {rec.id}")
    print(f"Sequence:  {rec.seq[:60]}...")
    print(f"Length:    {len(rec.seq)} bp")
    
    quals = rec.letter_annotations['phred_quality']
    print(f"Quality:   {quals[:20]}...")
    print(f"Mean Q:    {np.mean(quals):.1f}")
    
    # GC content
    gc = (rec.seq.count('G') + rec.seq.count('C')) / len(rec.seq) * 100
    print(f"GC:        {gc:.1f}%")
    
    # Summary statistics across all reads
    print("\nSummary across all reads:")
    all_quals = [np.mean(r.letter_annotations['phred_quality']) for r in records]
    all_lengths = [len(r.seq) for r in records]
    print(f"  Mean quality: Q{np.mean(all_quals):.1f}")
    print(f"  Mean length:  {np.mean(all_lengths):.0f} bp")

except ImportError:
    print("BioPython not installed. Install with: pip install biopython")
    print("\nUsing our custom parser instead:")
    for header, seq, qual in list(parse_fastq(tmpfile.name, max_reads=3)):
        print(f"  {header}: {seq[:40]}... Q={np.mean([ascii_to_phred(c) for c in qual]):.1f}")

os.unlink(tmpfile.name)  # clean up

In [ ]:
def quality_filter_reads(reads, min_mean_quality=25, min_length=50):
    """Filter reads by mean quality and minimum length."""
    passed = []
    failed = []
    
    for header, seq, qual in reads:
        mean_q = np.mean([ascii_to_phred(c) for c in qual])
        if mean_q >= min_mean_quality and len(seq) >= min_length:
            passed.append((header, seq, qual))
        else:
            failed.append((header, seq, qual))
    
    return passed, failed

def n_content_filter(reads, max_n_fraction=0.05):
    """Filter reads with too many N bases."""
    passed = []
    for header, seq, qual in reads:
        n_frac = seq.count('N') / len(seq)
        if n_frac <= max_n_fraction:
            passed.append((header, seq, qual))
    return passed

# Apply filters
passed, failed = quality_filter_reads(sim_reads, min_mean_quality=25)

print(f"Quality filter (mean Q >= 25):")
print(f"  Input:  {len(sim_reads)} reads")
print(f"  Passed: {len(passed)} ({100*len(passed)/len(sim_reads):.1f}%)")
print(f"  Failed: {len(failed)} ({100*len(failed)/len(sim_reads):.1f}%)")

# Quality comparison
before_q = [np.mean([ascii_to_phred(c) for c in q]) for _, _, q in sim_reads]
after_q = [np.mean([ascii_to_phred(c) for c in q]) for _, _, q in passed]
print(f"\n  Mean quality before: Q{np.mean(before_q):.1f}")
print(f"  Mean quality after:  Q{np.mean(after_q):.1f}")

---

## 10. Complete Pipeline Summary

```
Raw FASTQ reads
      |
      v
  FastQC (QC report)  <-- check quality, adapters, GC
      |
      v
  Trimming (fastp / Trimmomatic)  <-- remove adapters, low-quality bases
      |
      v
  FastQC (post-trim QC)  <-- verify improvement
      |
      v
  Alignment (BWA-MEM / HISAT2)  <-- map reads to reference genome
      |
      v
  SAM -> BAM (samtools view -bS)
      |
      v
  Sort BAM (samtools sort)
      |
      v
  Mark duplicates (samtools markdup / Picard)
      |
      v
  Index BAM (samtools index)
      |
      v
  QC: flagstat, coverage, depth
      |
      v
  Ready for downstream analysis:
    - Variant calling (GATK, bcftools)
    - Gene counting (featureCounts, HTSeq)
    - Peak calling (MACS2)
    - etc.
```

---

## Exercises

### Exercise 1: Phred Score Calculator

Write a function that takes a quality string and returns:
- Mean, median, min, and max Phred scores
- Number of bases with Q < 20
- Number of bases with Q >= 30

Test it on: `"IIIIIIIIII?????55555!!!!!!IIIII"`

In [ ]:
# Exercise 1: Your solution here

def quality_report(quality_string):
    """Analyze a quality string and return detailed statistics."""
    # YOUR CODE HERE
    pass

# Test
# quality_report("IIIIIIIIII?????55555!!!!!!IIIII")

### Exercise 2: CIGAR String Parser

Write a function that takes a CIGAR string and calculates:
- Number of aligned bases (M/=X)
- Number of inserted bases (I)
- Number of deleted bases (D)
- Number of soft-clipped bases (S)
- Alignment identity estimate: aligned / (aligned + insertions + deletions)

Test on: `"5S50M3I42M2D50M5S"`

In [ ]:
# Exercise 2: Your solution here

def alignment_stats(cigar_string):
    """Calculate detailed alignment statistics from a CIGAR string."""
    # YOUR CODE HERE
    pass

# Test
# alignment_stats("5S50M3I42M2D50M5S")

### Exercise 3: SAM Flag Decoder

Write a function that takes a SAM FLAG integer and returns a human-readable description including:
- Is the read paired?
- Is it mapped or unmapped?
- Which strand is it on?
- Is it R1 or R2?
- Is it a duplicate?

Test on flags: 99, 147, 4, 1024, 2064

In [ ]:
# Exercise 3: Your solution here

def describe_flag(flag):
    """Return a human-readable description of a SAM FLAG."""
    # YOUR CODE HERE
    pass

# Test
# for f in [99, 147, 4, 1024, 2064]:
#     print(f"FLAG {f}: {describe_flag(f)}")

### Exercise 4: Coverage Calculator

You are planning a sequencing experiment:
- Organism: *E. coli* (genome size = 4.6 Mb)
- Sequencer: Illumina MiSeq, 2x250 bp reads
- Target coverage: 100x

Calculate:
1. How many read pairs do you need?
2. How many total bases of sequence data?
3. What percentage of the genome will have >= 50x coverage?
4. If you instead use Oxford Nanopore with 10 kb average reads, how many reads for 100x?

In [ ]:
# Exercise 4: Your solution here

ecoli_genome = 4_600_000  # 4.6 Mb

# YOUR CODE HERE

### Exercise 5: Quality Trimming Implementation

Implement a complete read processor that:
1. Trims adapter sequences from the 3' end
2. Applies sliding window quality trimming (window=4, min_quality=20)
3. Trims leading bases below Q10
4. Trims trailing bases below Q10
5. Discards reads shorter than 50 bp

Apply it to the simulated reads and report:
- How many reads survived?
- What is the mean quality improvement?
- Plot the before/after per-position quality

In [ ]:
# Exercise 5: Your solution here

def process_read(sequence, quality_str, adapter='AGATCGGAAGAG'):
    """
    Complete read processing pipeline.
    Returns (trimmed_seq, trimmed_qual) or (None, None) if read is discarded.
    """
    # YOUR CODE HERE
    pass

# Apply to sim_reads and analyze results

### Exercise 6: Complete Pipeline Script

Write a Python script that processes a FASTQ file through the full pre-alignment pipeline:
1. Read the FASTQ file
2. Generate a QC summary (total reads, mean quality, GC content, length distribution)
3. Apply quality trimming
4. Generate a post-trim QC summary
5. Print a comparison table showing before/after statistics

Use the simulated reads as input.

In [ ]:
# Exercise 6: Your solution here

def full_qc_pipeline(reads):
    """
    Run complete QC pipeline on a list of (header, seq, qual) reads.
    """
    # YOUR CODE HERE
    pass

# full_qc_pipeline(sim_reads)

---

## Resources

### Tools
- [FastQC](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/) - Quality control
- [fastp](https://github.com/OpenGene/fastp) - Fast all-in-one preprocessing
- [Trimmomatic](http://www.usadellab.org/cms/?page=trimmomatic) - Read trimming
- [BWA](http://bio-bwa.sourceforge.net/) - Burrows-Wheeler Aligner
- [Bowtie2](http://bowtie-bio.sourceforge.net/bowtie2/) - Fast aligner
- [HISAT2](http://daehwankimlab.github.io/hisat2/) - Splice-aware aligner
- [STAR](https://github.com/alexdobin/STAR) - Splice-aware aligner
- [samtools](http://www.htslib.org/) - SAM/BAM manipulation
- [minimap2](https://github.com/lh3/minimap2) - Long-read aligner

### Specifications
- [SAM Format Specification](https://samtools.github.io/hts-specs/SAMv1.pdf)
- [FASTQ Format (Wikipedia)](https://en.wikipedia.org/wiki/FASTQ_format)

### Tutorials
- [Data Carpentry Genomics](https://datacarpentry.org/genomics-workshop/)
- [GATK Best Practices](https://gatk.broadinstitute.org/hc/en-us/sections/360007226651-Best-Practices-Workflows)
- [Galaxy Training: NGS](https://training.galaxyproject.org/training-material/topics/sequence-analysis/)

---[< Previous: Ready for Applied Bioinformatics?](../00_Skills_Check/00_skills_check.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Bioinformatics Data Formats: A Comprehensive Guide >](02_bio_data_formats.ipynb)---